# 13 - Multiple Cross-Model Readers: Analysis

Compare legibility scores across all reader models (Gemma, Llama, Mistral).
Report mean and variance of L across reader models.

In [ ]:
import subprocess, sys
from pathlib import Path

WORKSPACE = Path("/workspace/13-4-2026")
REPO_DIR = WORKSPACE / "legibility"

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
else:
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "git", "clone", "-b", "13-4-2026",
        "https://github.com/JackHopkins/legibility.git",
        str(REPO_DIR)
    ], check=True)

sys.path.insert(0, str(REPO_DIR))
from lib.config import *

for d in [CACHE_DIR, COT_CACHE, PARAPHRASE_CACHE, PREFILL_CACHE, RESULTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

def get_acc(condition):
    results = []
    for p in PREFILL_CACHE.glob(f"{condition}_*.json"):
        r = json.loads(p.read_text())
        results.append(r["predicted_answer"] == r["gold_answer"])
    return np.mean(results) if results else None

def bootstrap_ci(condition, n=10000, seed=42):
    results = []
    for p in PREFILL_CACHE.glob(f"{condition}_*.json"):
        r = json.loads(p.read_text())
        results.append(r["predicted_answer"] == r["gold_answer"])
    if not results:
        return None, None
    arr = np.array(results, dtype=float)
    rng = np.random.RandomState(seed)
    boots = sorted([rng.choice(arr, size=len(arr), replace=True).mean() for _ in range(n)])
    return boots[int(0.025*n)], boots[int(0.975*n)]

def reader_tag(name):
    return name.split("/")[-1].lower().replace("-", "_").replace(".", "")

# Primary model baselines
self_prefill_acc = get_acc("self_prefill")
no_cot_primary = get_acc("no_cot")
total_value_primary = self_prefill_acc - no_cot_primary

In [ ]:
# Compute L for each reader
reader_results = []

# Include original Gemma from core experiment
gemma_cp = get_acc("cross_prefill")
gemma_pc = get_acc("paraphrase_cross")
gemma_L = ((gemma_pc - no_cot_primary) / total_value_primary
           if gemma_pc and total_value_primary > 0 else None)
reader_results.append({
    "reader": CROSS_MODEL,
    "no_cot": no_cot_primary,  # Use primary no_cot as reference
    "cross_prefill": gemma_cp,
    "paraphrase_cross": gemma_pc,
    "L": gemma_L,
})

for reader_model in CROSS_READER_MODELS:
    tag = reader_tag(reader_model)
    if reader_model == CROSS_MODEL:
        continue  # Already added above

    nc = get_acc(f"no_cot_{tag}")
    cp = get_acc(f"cross_prefill_{tag}")
    pc = get_acc(f"paraphrase_cross_{tag}")

    # L relative to primary model's self_prefill and no_cot
    L = ((pc - no_cot_primary) / total_value_primary
         if pc is not None and total_value_primary > 0 else None)

    reader_results.append({
        "reader": reader_model,
        "no_cot_reader": nc,
        "cross_prefill": cp,
        "paraphrase_cross": pc,
        "L": L,
    })

for r in reader_results:
    print(f"{r['reader']:45s} | cp={r['cross_prefill']} | pc={r['paraphrase_cross']} | L={r['L']}")

# Save
with open(RESULTS_DIR / "multi_reader_results.json", "w") as f:
    json.dump(reader_results, f, indent=2, default=str)

In [ ]:
# Plot: L across readers
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

valid = [r for r in reader_results if r["L"] is not None]
names = [r["reader"].split("/")[-1] for r in valid]
l_vals = [r["L"] for r in valid]
cp_vals = [r["cross_prefill"] for r in valid]
pc_vals = [r["paraphrase_cross"] for r in valid]

colors = ["#27ae60", "#e67e22", "#2980b9"]

# Left: L scores
bars = ax1.bar(range(len(valid)), l_vals, color=colors[:len(valid)],
               edgecolor="black", linewidth=0.5)
for bar, val in zip(bars, l_vals):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f"{val:.3f}", ha="center", fontsize=10)

ax1.set_xticks(range(len(valid)))
ax1.set_xticklabels(names, rotation=15, fontsize=9)
ax1.set_ylabel("Legibility score (L)")
ax1.set_title("Legibility by Reader Model", fontweight="bold")
ax1.set_ylim(0, 1.5)

if len(l_vals) > 1:
    mean_L = np.mean(l_vals)
    std_L = np.std(l_vals)
    ax1.axhline(y=mean_L, color="blue", linestyle="-", alpha=0.3,
                label=f"Mean L={mean_L:.3f} (std={std_L:.3f})")
    ax1.legend()

# Right: cross_prefill vs paraphrase_cross grouped bar
x = np.arange(len(valid))
w = 0.35
ax2.bar(x - w/2, cp_vals, w, label="Cross prefill", color="#9b59b6", edgecolor="black", linewidth=0.5)
ax2.bar(x + w/2, pc_vals, w, label="Paraphrase cross", color="#27ae60", edgecolor="black", linewidth=0.5)
ax2.axhline(y=no_cot_primary, color="gray", linestyle="--", alpha=0.5, label=f"No-COT ({no_cot_primary:.3f})")
ax2.axhline(y=self_prefill_acc, color="purple", linestyle="--", alpha=0.5, label=f"Self-prefill ({self_prefill_acc:.3f})")
ax2.set_xticks(x)
ax2.set_xticklabels(names, rotation=15, fontsize=9)
ax2.set_ylabel("Accuracy")
ax2.set_title("Cross-Model Accuracy by Reader", fontweight="bold")
ax2.set_ylim(0, 1)
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "multi_reader.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Summary statistics
if l_vals:
    print(f"Legibility L across {len(l_vals)} readers:")
    print(f"  Mean:  {np.mean(l_vals):.3f}")
    print(f"  Std:   {np.std(l_vals):.3f}")
    print(f"  Min:   {np.min(l_vals):.3f}")
    print(f"  Max:   {np.max(l_vals):.3f}")
    print(f"  Range: {np.max(l_vals) - np.min(l_vals):.3f}")
    if np.std(l_vals) < 0.1:
        print("\nConclusion: L is STABLE across readers. The metric captures universal legibility.")
    else:
        print("\nConclusion: L VARIES across readers. The metric may reflect pairwise compatibility.")